# 00 · Pipeline walkthrough — data → prompt → model → parse → metrics

One runnable tour of the **whole KIE pipeline** on a single document, then the full
study. Change the knobs in the **Configure** cell and re-run top-to-bottom to inspect
any document, model, or condition.

```
config.yaml → load record → preprocess (simple JSON) → build_prompt → MODEL RUNNER
            → parse → metrics (strict + normalized) → [full run] → runs.jsonl → summary
```

**Requirements**
- **Local model** (e.g. `qwen2.5-0.5b`, `phi4-mini-ft`): needs `ollama serve` running and the
  tag pulled (`ollama list`).
- **Cloud model** (e.g. `deepseek-v3`, `gemma3-27b`, `qwen3-235b`): needs keys in `.env`
  (`AWS_BEARER_TOKEN_BEDROCK`). Cloud calls are **billed** per token.


## 0 · Setup

In [1]:
import sys, os, json
from pathlib import Path

# Make `src` importable and load API keys from .env (for cloud models).
ROOT = Path.cwd()
ROOT = ROOT if (ROOT / "src").exists() else ROOT.parent   # tolerate running from notebooks/
sys.path.insert(0, str(ROOT))
env = ROOT / ".env"
if env.exists():
    for line in env.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1); os.environ.setdefault(k.strip(), v.strip())

from src.config import load_config
from src.data.gt import load_test_records, load_test_set, align_gold, few_shot_pool, schema_for_record
from src.data.preprocess import from_dataset_record
from src.prompts.builder import build_prompt, json_schema_format, field_spec
from src.models.registry import build_runners
from src.eval.parse import parse_prediction
from src.eval.metrics import score_document

cfg = load_config()
print("config loaded. models:", [m["id"] for m in cfg["models"]])

config loaded. models: ['smollm2-360m', 'qwen2.5-0.5b', 'llama3.2-1b', 'gemma3-1b', 'gemma2-2b', 'gemma3-4b', 'gemma4-e2b', 'gemma4-e4b', 'phi4-mini', 'phi4-mini-ft', 'mistral-7b', 'mistral-7b-ft', 'gemma3-27b', 'deepseek-v3', 'qwen3-235b', 'llama-70b', 'llama4-maverick']


## 1 · Configure — pick a document, a model, and a condition
Change these and re-run the notebook. `MODEL_ID` can be any id printed above.

In [2]:
MODEL_ID      = "qwen2.5-0.5b"     # local default (free). Try "deepseek-v3" / "gemma3-27b" (cloud, billed).
DOC_INDEX     = 0                  # which test document (0 .. len(records)-1)
SHOT_MODE     = "zero_shot"        # "zero_shot" | "few_shot"
INPUT_VARIANT = "lines_only"       # "lines_only" | "lines_plus_kv"

records = load_test_records(cfg)
max_lines = cfg["conditions"].get("max_input_lines")
few_shot_k = int(cfg["conditions"]["few_shot_k"])
structured = cfg["ollama"].get("structured_output", False)

# dataset composition of the frozen 20-doc test set
from collections import Counter
print("test set:", dict(Counter(r["dataset"] for r in records)), "| total", len(records))
print(f"selected: doc #{DOC_INDEX}  model={MODEL_ID}  shot={SHOT_MODE}  variant={INPUT_VARIANT}")

test set: {'sroie': 10, 'kleister_charity': 10} | total 20
selected: doc #0  model=qwen2.5-0.5b  shot=zero_shot  variant=lines_only


## 2 · See the data — one document (raw record + preprocessed 'simple JSON')

In [3]:
rec = records[DOC_INDEX]
schema = schema_for_record(cfg, rec)
gold = align_gold(rec.get("gold", {}), schema)
simple_json = from_dataset_record(rec)   # the canonical shape every backend consumes

print("dataset :", rec["dataset"], "| doc_id:", rec["doc_id"])
print("schema  :", schema["dataset"], "-", [f["name"] for f in schema["fields"]])
print("\nGOLD (target fields to extract):")
print(json.dumps(gold, indent=2, ensure_ascii=False))
print("\nSIMPLE JSON — first 15 of", len(simple_json.get("lines", [])), "document lines:")
for ln in simple_json.get("lines", [])[:15]:
    print("   ", ln)
kv = simple_json.get("key_values", {})
print("\nkey_values:", dict(list(kv.items())[:6]), "..." if len(kv) > 6 else "")

dataset : sroie | doc_id: 132
schema  : sroie - ['company', 'date', 'address', 'total']

GOLD (target fields to extract):
{
  "company": "FARMASI MALURI S/B",
  "date": "02/03/18",
  "address": "23, JIN BURUNG JENTAYU, TMN BUKIT MALURI, KEPONG, 52100 KL.",
  "total": "79.35"
}

SIMPLE JSON — first 15 of 28 document lines:
    3180301
    FARMASI MALURI S/B (587969-U)
    GST NO: 000342237184 (ID:5033)
    23, JIN BURUNG JENTAYU, TMN BUKIT
    MALURI, KEPONG, 52100 KL.
    TAX INVOICE
    CASHIER : POS1N M/C. ID :POS7
    RECEIPT : 71857936 02/03/18 05:16 PM
    DESCRIPTION DISC. SUBTOTAL
    061558 - **BMS MILK THISTLE
    1 X 111.30 36.75 72.35 S
    910845 - TRILON-E CRM
    1 X 7.00 7.00 Z
    TOTAL (INCLUDING GST) 79.35
    ROUNDING 0.00

key_values: {} 


## 3 · The exact prompt fed to the model
This is the **verbatim** string the model sees — instruction + field spec (with hints) +
optional few-shot examples (drawn from the **train** split only) + the document.

In [4]:
test_doc_ids = {s["doc_id"] for s in load_test_set(cfg)["doc_ids"]}
examples = few_shot_pool(cfg, rec, test_doc_ids, few_shot_k) if SHOT_MODE == "few_shot" else None

prompt = build_prompt(schema, simple_json, SHOT_MODE, INPUT_VARIANT, examples, max_lines=max_lines)
print(f"[prompt length: {len(prompt)} chars, ~{len(prompt)//4} tokens]\n" + "="*80)
print(prompt)

[prompt length: 1560 chars, ~390 tokens]
You are a precise information-extraction system for financial documents.

You are given the text of a document (its lines, and optionally detected
key/value pairs). Extract the following fields and return them as a single JSON
object with EXACTLY these keys:

- "company" (string, required) — the merchant/business name only; exclude registration numbers in parentheses
- "date" (date, required) — the receipt/transaction date
- "address" (string, required) — the merchant's full street address as printed
- "total" (currency, required) — the final total amount paid

Rules:
- Return ONLY the JSON object. No explanation, no markdown, no code fences.
- Use the exact key names listed above.
- If a field cannot be found in the document, set its value to null.
- Copy values verbatim from the document where possible; do not invent data.


### Document
LINES:
3180301
FARMASI MALURI S/B (587969-U)
GST NO: 000342237184 (ID:5033)
23, JIN BURUNG JENTAYU, TMN BUK

## 4 · Run the model → raw output
Local models get JSON-schema-constrained decoding (`response_format`); cloud models use
prompt-instructed JSON. Same interface for both — `runner.run(prompt, response_format)`.

In [5]:
runner = build_runners(cfg, only=[MODEL_ID])[0]
print("runner:", type(runner).__name__, "| kind:", runner.kind)
runner.ensure_available()   # local: checks Ollama + tag; cloud: checks creds

response_format = json_schema_format(schema) if (structured and runner.kind == "local") else None
result = runner.run(prompt, response_format)

print("\n--- RAW MODEL OUTPUT ---")
print(result.text)
print("\nlatency: %.2fs | prompt_tok: %s | completion_tok: %s | error: %s" %
      (result.latency_s or 0, result.prompt_tokens, result.completion_tokens, result.error))

runner: OllamaRunner | kind: local



--- RAW MODEL OUTPUT ---
{
  "company": "FARMASI MALURI",
  "date": "2018-03-03",
  "address": "JIN BURUNG JENTAYU, TMN BUKIT MELAKAN, KEPONG, 52100 KL.",
  "total": "79.35"
}

latency: 4.04s | prompt_tok: 592 | completion_tok: 78 | error: None


## 5 · Parse the raw text → predicted JSON (defensive; never raises)

In [6]:
pred, parse_err = parse_prediction(result.text, schema)
print("parse_error:", parse_err)
print("\n%-34s %-34s" % ("PREDICTED", "GOLD"))
print("-"*70)
for f in [f["name"] for f in schema["fields"]]:
    p = (pred or {}).get(f); g = gold.get(f)
    print("%-16s %-17s %-16s %s" % (f[:16], repr(p)[:17], "", repr(g)[:20]))

parse_error: None

PREDICTED                          GOLD                              
----------------------------------------------------------------------
company          'FARMASI MALURI'                   'FARMASI MALURI S/B'
date             '2018-03-03'                       '02/03/18'
address          'JIN BURUNG JENTA                  '23, JIN BURUNG JENT
total            '79.35'                            '79.35'


## 6 · Metrics calculation — strict (exact) vs normalized (Lever 2)
Per field → one category:
`tp` (match) · `wrong` (both present, differ) · `missing` (gold present, pred null, a recall miss) ·
`hallucinated` (gold null, pred present) · `tn` (both null).
**precision = tp/(tp+wrong+hallucinated)** · **recall = tp/(tp+wrong+missing)** ·
**F1 = harmonic mean**. `normalized` ignores punctuation/whitespace (applied identically to
gold+pred), reported alongside strict.

In [7]:
strict  = score_document(gold, pred, schema)
lenient = score_document(gold, pred, schema, lenient=True)

print("per-field category (strict):")
for f, c in strict["per_field"].items():
    flip = "  <- would match under normalized" if (strict["per_field"][f]=="wrong" and lenient["per_field"][f]=="tp") else ""
    print(f"   {f:34} {c}{flip}")
print("\ncounts (strict):", strict["counts"])
print("\n%-12s %8s %8s" % ("metric", "strict", "normalized"))
for k in ("precision", "recall", "f1"):
    print("%-12s %8.3f %8.3f" % (k, strict[k], lenient[k]))
print("exact_match_doc:", strict["exact_match_doc"], "| fields correct:",
      strict["exact_match_fields"], "/", strict["n_fields"])

per-field category (strict):
   company                            wrong
   date                               wrong
   address                            wrong
   total                              tp

counts (strict): {'tp': 1, 'wrong': 3, 'missing': 0, 'hallucinated': 0, 'tn': 0}

metric         strict normalized
precision       0.250    0.250
recall          0.250    0.250
f1              0.250    0.250
exact_match_doc: False | fields correct: 1 / 4


## 7 · Run the whole study & save results
Sections 1–6 are exactly one cell of the grid. `run_eval` scales it over
**docs × models × shot_modes × input_variants**, writes one JSON row per cell to
`results/runs.jsonl` (**resumable** — done cells are skipped), and is the same code path.

⚠️ The full run covers every model; **cloud models are billed**. Start scoped.

In [8]:
from src.runner.run import run_eval

RUN = False   # <- set True to actually run. Scoped to MODEL_ID on a few docs below.

if RUN:
    summary = run_eval(only_models=[MODEL_ID], pilot=True)   # pilot = few docs, zero-shot
    print("done:", summary)
    print("results appended to:", cfg["paths"]["runs_jsonl"])
else:
    print("RUN is False — flip to True to execute. Options:")
    print("  run_eval(only_models=[MODEL_ID], pilot=True)     # cheapest: few docs")
    print("  run_eval(only_models=[MODEL_ID])                 # full 80-cell grid, one model")
    print("  run_eval()                                       # EVERYTHING (all models; billed)")

RUN is False — flip to True to execute. Options:
  run_eval(only_models=[MODEL_ID], pilot=True)     # cheapest: few docs
  run_eval(only_models=[MODEL_ID])                 # full 80-cell grid, one model
  run_eval()                                       # EVERYTHING (all models; billed)


### Analyze the saved results
After a run, regenerate the tables/plots and the strict-vs-normalized comparison:
```bash
python scripts/make_plots.py          # -> results/summary.csv, PNGs, results/REPORT.md
python scripts/lenient_rescore.py     # strict vs normalized F1 per model
```

In [9]:
import pandas as pd
summ = ROOT / "results" / "summary.csv"
if summ.exists():
    df = pd.read_csv(summ)
    display(df.head(20))
else:
    print("No summary.csv yet — run scripts/make_plots.py after an eval.")

,model_id,model_type,shot_mode,input_variant,n,f1_macro_mean,f1_macro_std,f1_micro,precision_micro,recall_micro,exact_match_rate,latency_s_mean,tokens_per_s_mean,peak_mem_mb_mean,cost_usd_per_doc,parse_fail_rate,missing,hallucinated
0,deepseek-v3,api,few_shot,lines_only,20,0.7258,0.2231,0.7064,0.7857,0.6417,0.25,1.963,46.80,NaN,0.001332,0.0,22,0
1,deepseek-v3,api,few_shot,lines_plus_kv,20,0.7207,0.2225,0.6912,0.7732,0.6250,0.25,1.900,47.59,NaN,0.001344,0.0,23,0
2,deepseek-v3,api,zero_shot,lines_only,20,0.6887,0.2315,0.6728,0.7526,0.6083,0.20,1.987,44.51,NaN,0.000718,0.0,23,0
3,deepseek-v3,api,zero_shot,lines_plus_kv,20,0.6652,0.2130,0.6452,0.7216,0.5833,0.20,2.039,44.43,NaN,0.000721,0.0,23,0
4,gemma2-2b,local,few_shot,lines_only,20,0.6377,0.2175,0.6009,0.6195,0.5833,0.15,13.439,8.75,2401.8,0.000000,0.0,7,0
5,gemma2-2b,local,few_shot,lines_plus_kv,20,0.6619,0.2246,0.6160,0.6239,0.6083,0.20,13.399,8.75,2573.7,0.000000,0.0,3,0
6,gemma2-2b,local,zero_shot,lines_only,20,0.5711,0.2268,0.5472,0.6304,0.4833,0.05,11.993,8.83,2469.8,0.000000,0.0,28,0
7,gemma2-2b,local,zero_shot,lines_plus_kv,20,0.5746,0.2231,0.5514,0.6277,0.4917,0.05,6.429,15.00,2201.8,0.000000,0.0,26,0
8,gemma3-1b,local,few_shot,lines_only,20,0.4188,0.2718,0.3500,0.3500,0.3500,0.05,4.633,24.78,1624.3,0.000000,0.0,0,0
9,gemma3-1b,local,few_shot,lines_plus_kv,20,0.3938,0.2280,0.3333,0.3333,0.3333,0.00,4.619,24.72,1639.7,0.000000,0.0,0,0
